# Erweitertes Mathematisches Modell: Hazmat-CVRP-MT mit Risiko-, Kosten- und Zeitoptimierung

---

## 0. Änderungen gegenüber dem Ausgangsmodell — Überblick

| Bereich | Ausgangsmodell | Erweitertes Modell | Begründung |
|---|---|---|---|
| Lieferstruktur | 1 Lieferung = 1 Start, 1 Ziel | VRP-Touren mit mehreren Kunden pro Fahrzeug | Bula et al. (2016); Androutsopoulos & Zografos (2012) |
| Fahrzeugeinsatz | 1 Tour pro Fahrzeug | Multi-Trip (mehrere Touren/Tag, Depot-Rückkehr, Neubeladung) | reale Flottenverfügbarkeit |
| Risiko Hin-/Rückfahrt | symmetrisch | nur Hinfahrt (beladen); Rückfahrt risikofrei, frei geroutet | List & Mirchandani (1988); lastabhängiges Risiko nach Bula et al. (2016) |
| ADR-Autobahnpflicht | nicht modelliert | Penalty-Ansatz in der Vorverarbeitung (Dijkstra-Kostenfunktion) | reales Fahrverhalten |
| Tunnelrestriktionen | generische Sperrkanten | explizite Tunnelkategorien A–D je Gefahrgutklasse | ADR/GGVSEB |
| Zeitkomponente | nicht modelliert | vollständige Zeitfortschreibung inkl. Service-, Lade- und Pausenzeiten | Bell (2006): Routing & Scheduling sind nicht trennbar |
| Lenk-/Ruhezeiten | nicht modelliert | VO (EG) 561/2006 + ArbZG als harte Nebenbedingungen | gesetzliche Pflicht, vom Auftraggeber explizit gefordert |
| Problemgröße | vollständiger Straßengraph im Solver | zweistufige Architektur: Dijkstra-Reduktion → VRP-MILP auf 21 Knoten | Bula et al. (2016); Toth & Vigo (2002, Standardreferenz für den Distanzmatrix-Ansatz) |
| Solver-Strategie | ein Solver (CBC) | exaktes MILP für Prototyp, OR-Tools/ALNS für Deutschland-Skala | Androutsopoulos & Zografos (2012): exakte Solver scheitern bereits bei vergleichbarer Größenordnung |

Das Modell ist jetzt zweistufig aufgebaut: **Ebene 1** reduziert den realen Straßengraphen (bzw. später ganz Deutschland) auf eine Distanz-/Risiko-/Zeitmatrix zwischen den 21 relevanten Knoten (Depot + 20 Kunden + ggf. Ladeknoten). **Ebene 2** löst auf dieser reduzierten Matrix ein klassisches, aber stark erweitertes Multi-Trip-Hazmat-VRP.

---

## Ebene 1 — Vorverarbeitung: Graphreduktion

Diese Ebene war im Ausgangsmodell bereits konzeptionell vorhanden ("Graphreduktion implementiert"), wird hier aber explizit formalisiert, weil sie jetzt zwei unterschiedliche Kostenfunktionen pro Knotenpaar liefern muss.

**Eingabe:** realer Straßengraph $G_{road} = (N_{road}, E_{road})$ mit Distanzen, Risikodaten, Tunnelklassifikation und Autobahnkennzeichnung je Kante.

Für jedes Knotenpaar $(i,j)$ mit $i,j \in N = \{0\} \cup C \cup H$ (Depot, Kunden, relevante HPC-Ladeknoten) werden **zwei** kürzeste Pfade berechnet:

**(a) Hinfahrt-Pfad** (beladen, Klasse $k$): Dijkstra mit modifizierter Kantenkostenfunktion

$$
\widehat{Cost}_{e} = w_1 \cdot Risk_{e,k} + w_2 \cdot VC_{e} + \underbrace{\Pi_e}_{\text{Autobahn-Penalty}}, \qquad
\Pi_e = \begin{cases} \pi & \text{falls } e \notin \text{Autobahn} \wedge Class_k = 3 \wedge Dem \geq 0{,}25\,t \\ 0 & \text{sonst} \end{cases}
$$

vorab werden alle Kanten entfernt, die laut Tunnelkategorie für Klasse $k$ gesperrt sind (List & Mirchandani, 1988: Vorab-Screening zur Entfernung ungeeigneter Strecken). $\pi$ wird so groß gewählt, dass der Solver Autobahnabschnitte immer dann nutzt, wenn eine Alternative existiert, und nur auf den letzten Kilometern zum Kunden abweicht.

Daraus ergeben sich die Parameter $Dist^{out}_{i,j}$, $RiskRaw_{i,j,k}$, $Time^{out}_{i,j}$, $HwShare_{i,j}$.

**(b) Rückfahrt-Pfad** (leer, ohne Gefahrgut): einfacher kürzester Pfad nach Distanz/Zeit, ohne Tunnel- oder Autobahnrestriktion, ohne Risikoterm — da die transportierte Menge auf der Rückfahrt null ist, ist das Schadensrisiko nach der im Paper von List & Mirchandani (1988) verwendeten Definition (Risiko als Funktion der Ladungsmenge) ebenfalls null.

Daraus ergeben sich $Dist^{ret}_{i,0}$, $Time^{ret}_{i,0}$ — diese Parameter werden **ausschließlich** für Kanten verwendet, die zum Depot zurückführen.

Bei 21 Knoten ergeben sich $21 \times 20 = 420$ gerichtete Hinfahrt-Läufe plus die Rückfahrt-Sonderläufe — in der Größenordnung der von euch genannten 441 Dijkstra-Läufe.

> **Für Deutschland-Maßstab:** Dieselbe Logik, aber Ebene 2 wird nicht mehr exakt gelöst, sondern an OR-Tools übergeben (siehe Abschnitt "Lösungsverfahren").

---

## Ebene 2 — Mengen (Sets)

| Symbol | Bedeutung |
|---|---|
| $V$ | Fahrzeuge, $\lvert V \rvert = 5$, heterogen: $V = V_{klein} \cup V_{mittel} \cup V_{groß}$, $\lvert V_{klein}\rvert=2, \lvert V_{mittel}\rvert=2, \lvert V_{groß}\rvert=1$ |
| $C$ | Kunden, $\lvert C \rvert = 20$ |
| $H$ | relevante HPC-Ladeknoten (Teilmenge der für die betrachteten Touren erreichbaren Stationen) |
| $N$ | $N = \{0\} \cup C \cup H$, Knoten $0$ = Depot |
| $K$ | Gefahrgutklassen (primär Klasse 3, optional Klasse 2) |
| $R$ | mögliche Touren-Indizes je Fahrzeug und Tag, $R = \{1, \dots, R_{max}\}$ (Multi-Trip) |

---

## Parameter

### Unverändert / direkt übernommen
$Cap_v$, $FC_v$, $c^{km}_v$, $\varepsilon_v$, $Range_v$, $Dem_c$, $Class_c$, $Allow_{e,k}$, $ChargeNode_n$, $p_e$ (Energiepreis nach Kantentyp)

### Neu / aktualisiert

| Symbol | Einheit | Bedeutung |
|---|---|---|
| $Dist^{out}_{i,j}$, $Dist^{ret}_{i,0}$ | km | Distanzen aus Ebene 1, getrennt nach Hin-/Rückfahrt |
| $Time^{out}_{i,j}$, $Time^{ret}_{i,0}$ | h | reine Fahrzeit, getrennt nach Hin-/Rückfahrt |
| $Risk_{i,j,k}$ | — | $Risk_{i,j,k} = \mu_k \cdot \big(\alpha \cdot \text{PopDens}_{i,j} + \beta \cdot \text{AccRate}_{i,j} + \gamma \cdot \text{NatRes}_{i,j}\big)$, normiert auf $[0,1]$; **nur für Hinfahrt-Kanten definiert** |
| $\alpha, \beta, \gamma$ | — | $0{,}40 / 0{,}40 / 0{,}20$ (Cuneo et al., 2018, als Best-Practice-Gewichtung für Kraftstofflogistik übernommen; Naturschutz als methodische Erweiterung nach List & Mirchandani, 1988) |
| $\mu_k$ | — | Gefahrgutklassen-Multiplikator (Klasse 3 vs. Klasse 2 erzeugen laut Bula et al., 2016, unterschiedliche optimale Routen) |
| $ServiceTime_c$ | h | $= 0{,}75$ (45 Min., pauschal je Kunde) |
| $ChargeDuration_v$ | h | Ladezeit von Fahrzeug $v$ an einer HPC-Station |
| $ReloadTime_v$ | h | Beladezeit am Depot zwischen zwei Touren |
| $HwyThreshold_k$ | t | Gewichtsschwelle für Autobahnpflicht; für Klasse 3 (Benzin): $333\,l \times 0{,}75\,kg/l \approx 0{,}25\,t$ |
| $DrivingLimit$ | h | $4{,}5$ (max. Lenkzeit bis Pause, VO (EG) 561/2006) |
| $BreakDuration$ | h | $0{,}75$ (45 Min., alternativ $0{,}25 + 0{,}50$ in dieser Reihenfolge) |
| $DailyDrivingMax$ | h | $9$ (Regel), $10$ (Ausnahme, zulässig $\leq 2\times$/Woche — im Prototyp als Parameter, nicht als Wochenzähler) |
| $DailyWorkMax$ | h | $8$ (Regel), $10$ (Ausnahme, ArbZG) |
| $RestPeriod$ | h | $11$ (reduzierbar auf $9$, ArbZG/VO 561/2006) |
| $M$ | — | hinreichend großes Big-M, fahrzeug-/touren-spezifisch über $|N|$, $DailyWorkMax$ bzw. $\sum Range_v$ gesetzt |

---

## Entscheidungsvariablen

| Variable | Typ | Bedeutung |
|---|---|---|
| $x_{v,r,i,j} \in \{0,1\}$ | Binär | 1, wenn Fahrzeug $v$ auf Tour $r$ die Kante $(i,j)$ befährt |
| $y_{v,r,c} \in \{0,1\}$ | Binär | 1, wenn Kunde $c$ auf Tour $r$ von Fahrzeug $v$ beliefert wird |
| $z_{v,r} \in \{0,1\}$ | Binär | 1, wenn Fahrzeug $v$ Tour $r$ überhaupt durchführt |
| $ch_{v,r,n} \in \{0,1\}$ | Binär | 1, wenn auf Tour $r$ von $v$ an Knoten $n$ geladen wird |
| $brk_{v,r,n} \in \{0,1\}$ | Binär | 1, wenn an Knoten $n$ auf Tour $r$ von $v$ die Lenkpause eingelegt wird |
| $u_{v,r,n} \geq 0$ | Stetig | MTZ-Hilfsvariable je Fahrzeug-Tour |
| $q_{v,r,n} \geq 0$ | Stetig | Restreichweite (km) bei Ankunft an Knoten $n$ |
| $ST_{v,r,n} \geq 0$ | Stetig | Ankunftszeit (Uhrzeit/Stunden seit Schichtbeginn) an Knoten $n$ |
| $DS_{v,r,n} \geq 0$ | Stetig | kumulierte Lenkzeit seit letzter Pause bei Ankunft an Knoten $n$ |
| $CostVar_{v,r} \geq 0$ | Stetig | exakte variable Kosten der Tour $r$ von Fahrzeug $v$ |

---

## Zielfunktion

Drei Kriterien werden gemeinsam minimiert — methodisch gedeckt durch Androutsopoulos & Zografos (2012), die eine gewichtete Summe als valide Methode für bi-/multikriterielle Hazmat-Routenprobleme begründen:

$$
\min \quad
\frac{w_1}{Risk_{\max}} \sum_{v,r} \sum_{(i,j):\, j \neq 0} Risk_{i,j,k(c)} \cdot x_{v,r,i,j}
\;+\;
\frac{w_2}{Cost_{\max}} \left( \sum_v FC_v \cdot \sum_r z_{v,r} + \sum_{v,r} CostVar_{v,r} \right)
\;+\;
\frac{w_3}{Time_{\max}} \sum_{v,r} \big( ST_{v,r,0}^{return} - ST_{v,r,0}^{start} \big)
$$

Der Risikoterm läuft explizit nur über Kanten mit $j \neq 0$ (also über alle Hinfahrt- bzw. Zwischenkanten), da Rückfahrtkanten $(i,0)$ per Konstruktion aus Ebene 1 keinen Risikowert besitzen (List & Mirchandani, 1988). $w_1, w_2, w_3$ sind frei wählbar; mit $w_3 = 0$ reduziert sich das Modell auf das ursprüngliche bikriterielle Risiko-Kosten-Modell, was für Sensitivitätsanalysen sinnvoll ist (Bell, 2006: Variation der Gewichte erzeugt unterschiedliche Punkte auf der Pareto-Front).

---

## Nebenbedingungen

### C1 — Kundenzuweisung
$$
\sum_{v \in V} \sum_{r \in R} y_{v,r,c} = 1 \qquad \forall c \in C
$$
Jeder Kunde wird in genau einer Fahrzeug-Tour-Kombination beliefert.

### C2 — Kopplung Kunde–Tour
$$
y_{v,r,c} \leq z_{v,r} \qquad \forall v, r, c
$$
Ein Kunde kann nur einer aktiven Tour zugeordnet sein.

### C3 — Kapazität je Tour
$$
\sum_{c \in C} Dem_c \cdot y_{v,r,c} \leq Cap_v \cdot z_{v,r} \qquad \forall v, r
$$

### C4 — Touren-Sequenz (Symmetriebrechung, Multi-Trip)
$$
z_{v,r} \leq z_{v,r-1} \qquad \forall v,\; r \in R,\, r>1
$$
Tour $r$ darf nur stattfinden, wenn Tour $r-1$ desselben Fahrzeugs bereits aktiv ist — verhindert äquivalente Permutationen und reduziert den Suchraum.

### C5/C6 — Grad-Constraints (Routing)
$$
\sum_{i \in N \setminus \{c\}} x_{v,r,i,c} = y_{v,r,c}, \qquad
\sum_{j \in N \setminus \{c\}} x_{v,r,c,j} = y_{v,r,c} \qquad \forall v,r,c
$$
Jeder belieferte Kunde hat genau eine eingehende und eine ausgehende Kante auf seiner Tour. Dies ersetzt die ursprüngliche Flusserhaltung (C5 im Ausgangsmodell), die für Einzellieferungen mit nur einer Quelle/Senke konzipiert war.

### C7 — Depot als Start- und Endpunkt
$$
\sum_{j \in N} x_{v,r,0,j} = z_{v,r}, \qquad \sum_{i \in N} x_{v,r,i,0} = z_{v,r} \qquad \forall v, r
$$

### C8 — Subtour-Eliminierung (MTZ, je Fahrzeug-Tour)
$$
u_{v,r,i} - u_{v,r,j} + |N| \cdot x_{v,r,i,j} \leq |N| - 1 \qquad \forall v,r,\, (i,j) \in N \times N,\, j \neq 0
$$
Funktionsweise identisch zum Ausgangsmodell, jetzt je Fahrzeug-Tour-Kombination statt je Lieferung.

### C9 — Gefahrgut- und Tunnelrestriktion
Bereits in Ebene 1 durch Entfernen gesperrter Kanten vor dem Dijkstra-Lauf umgesetzt (List & Mirchandani, 1988: Vorab-Screening). Zur Modelltransparenz dennoch redundant explizit gehalten:
$$
x_{v,r,i,j} \leq Allow_{(i,j),\, Class_c} \qquad \text{für } c \text{ mit } y_{v,r,c}=1
$$

### C10 — Ladezustand (SoC)
$$
q_{v,r,j} \leq q_{v,r,i} - Dist^{out}_{i,j} \cdot \mathbb{1}[j\neq 0] - Dist^{ret}_{i,j}\cdot \mathbb{1}[j=0] + M(1 - x_{v,r,i,j}) + M \cdot ch_{v,r,i} \qquad \forall v,r,(i,j)
$$

### C11 — Akkukapazität
$$
q_{v,r,n} \leq Range_v \qquad \forall v,r,n
$$

### C12 — Ladeinfrastruktur
$$
ch_{v,r,n} \leq ChargeNode_n \qquad \forall v,r,n
$$

### C13 — Zeitfortschreibung
$$
ST_{v,r,j} \geq ST_{v,r,i} + \underbrace{Time^{out}_{i,j} \cdot \mathbb{1}[j \neq 0] + Time^{ret}_{i,j} \cdot \mathbb{1}[j=0]}_{\text{Fahrzeit}} + \underbrace{ServiceTime_i \cdot y_{v,r,i}}_{\text{Service}} + \underbrace{ChargeDuration_v \cdot ch_{v,r,i}}_{\text{Laden}} + \underbrace{BreakDuration \cdot brk_{v,r,i}}_{\text{Pause}} - M(1-x_{v,r,i,j})
$$
Bell (2006) und die referenzierte Quelle zu Routing/Scheduling betonen, dass beide Planungsebenen nicht getrennt betrachtet werden dürfen — diese Nebenbedingung koppelt sie explizit in einer einzigen Zeitvariable je Knoten.

### C14 — Lenkzeitkumulierung
$$
DS_{v,r,j} \geq DS_{v,r,i} + Time^{out}_{i,j} - M \cdot brk_{v,r,i} - M(1-x_{v,r,i,j}) \qquad \forall v,r,(i,j)
$$
Wird an Knoten $i$ eine Pause eingelegt ($brk=1$), wird der Lenkzeit-Zähler auf null zurückgesetzt (analog zur Big-M-Logik von C10/C7 im Ausgangsmodell).

### C15 — Pausenpflicht (4,5-Stunden-Regel, VO (EG) 561/2006)
$$
DS_{v,r,n} \leq DrivingLimit \qquad \forall v,r,n
$$
In Kombination mit C14 erzwingt dies spätestens nach $4{,}5$ Stunden ununterbrochener Lenkzeit eine Pause von mindestens $45$ Minuten (vereinfachtes Modell; die zulässige Aufteilung $15+30$ Min. ist als spätere Erweiterung über zwei separate Pausenvariablen mit fester Reihenfolge vorgesehen).

### C16 — Tageslenkzeit
$$
\sum_{r \in R} \sum_{(i,j)} Time^{out}_{i,j} \cdot x_{v,r,i,j} \;+\; \sum_{r} \sum_{(i,0)} Time^{ret}_{i,0}\cdot x_{v,r,i,0} \leq DailyDrivingMax \qquad \forall v
$$

### C17 — Fahrzeugverfügbarkeit zwischen Touren (Multi-Trip-Kopplung)
$$
ST_{v,r+1,0}^{start} \geq ST_{v,r,0}^{return} + ReloadTime_v \cdot z_{v,r} \qquad \forall v,\; r \in R, r < R_{max}
$$
Formalisiert die Dispatch-Regel "Fahrzeug ist erst nach Rückkehr und Neubeladung wieder verfügbar". Für den exakten Berlin-Prototyp wird die optimale Touren-Fahrzeug-Zuordnung direkt mitoptimiert; für OR-Tools/ALNS im Deutschland-Maßstab wird stattdessen die einfachere First-Come-First-Served-Heuristik verwendet (siehe Lösungsverfahren).

### C18 — Kosten-Linearisierung
$$
CostVar_{v,r} \geq \sum_{(i,j):\,j\neq 0} VC_{v,(i,j)} \cdot x_{v,r,i,j} + \sum_{(i,0)} VC_{v,(i,0)} \cdot x_{v,r,i,0} - M(1-z_{v,r}) \qquad \forall v,r
$$
funktional identisch zu C10 im Ausgangsmodell, jetzt je Tour statt je Lieferung.

### C19 — Autobahnpflicht-Schwelle
Wird nicht als separate MILP-Nebenbedingung geführt, sondern bereits in Ebene 1 über den Penalty-Term $\Pi_e$ in der Dijkstra-Kostenfunktion durchgesetzt (siehe oben), aktiviert nur, wenn $Dem_c \geq HwyThreshold_k$. Diese Entkopplung hält das MILP in Ebene 2 linear und vermeidet zusätzliche Binärvariablen.

---

## Lösungsverfahren

| Maßstab | Verfahren | Begründung |
|---|---|---|
| Berlin-Prototyp (20 Kunden, 21 Knoten, 5 Fahrzeuge) | exaktes MILP, PuLP/CBC, mit Zeitlimit | Androutsopoulos & Zografos (2012): exakte Solver lösen Instanzen dieser Größenordnung in Sekunden bis Minuten (in ihrer Studie mit Gurobi 17 s – 30 min); CBC als schwächere Open-Source-Alternative benötigt ein Zeitlimit, um auch ohne Optimalitätsbeweis eine Lösung zu liefern |
| Deutschland (~10 Mio. Knoten im Straßengraph) | OR-Tools (Constraint-Programming-VRP-Solver) oder ALNS | Androutsopoulos & Zografos (2012) dokumentieren, dass bereits ein 100-Knoten-Netz mit 20 Kunden den exakten Solver (CPLEX) vollständig scheitern lässt; Metaheuristiken erreichen nach Bula et al. (2016) typischerweise 95–99 % der Optimallösung bei einem Bruchteil der Rechenzeit |

---

## Nicht implementierte, aber dokumentierte Erweiterungen (Ausblick)

Diese Punkte werden bewusst **nicht** in die aktuelle Modellversion aufgenommen, da sie die Komplexität deutlich erhöhen würden, aber als nächste Ausbaustufen festgehalten:

1. **Lastabhängiges Risiko** $Risk(s, q)$ als Potenzfunktion mit empirisch belegten Parametern für Benzin ($\alpha=0{,}72$, $P_{release}\approx0{,}025$, $TTAR\approx10^{-6}$/Fzg-km) statt statischer Kantengewichte (Bula et al., 2016).
2. **Mixed Route Strategies / Edge-Memory:** temporäre Penalty-Erhöhung auf bereits am selben Tag befahrenen Gefahrgutkanten, um Risikokonzentration auf einzelne Korridore zu vermeiden (Bell, 2006).
3. **Zeitabhängige Risiko- und Fahrzeitwerte** (Tagesverkehr, Bevölkerungsdichte-Schwankungen) statt statischer Tageswerte (Androutsopoulos & Zografos, 2012).
4. **Dynamische Ladekosten** an HPC-Stationen als zeitvariabler Term in $w_2$.
5. **Wöchentliche Lenk-/Ruhezeit-Aggregation** (56 h/Woche, 90 h/2 Wochen; 9-h-Ruhezeit max. dreimal zwischen zwei Wochenruhezeiten) — im aktuellen Modell nur auf Tagesebene abgebildet, da der Prototyp einen Einzeltag plant.
6. **Echtzeit-Replanning** bei Staus/Unfällen/Wetter — erfordert eine rollierende Neuoptimierung, die über das statische Tagesmodell hinausgeht.

---

## Modelltyp (aktualisiert)

- **Gemischt-ganzzahliges lineares Programm (MILP)**, zweistufig vorprozessiert
- **Heterogeneous Hazmat Vehicle Routing Problem mit Multi-Trip und Zeitfenstern** (Hazmat-HVRP-MT), eine Kombination aus EVRP (Ladeinfrastruktur, SoC-Tracking), Multi-Trip-VRP und gesetzlicher Lenkzeit-Planung
- Für den Prototyp gelöst mit PuLP/CBC; für Deutschland-Maßstab mit OR-Tools oder ALNS

---

## Literaturbezüge — Zusammenfassung

| Modellbestandteil | Quelle |
|---|---|
| Gewichtete Summe Risiko/Kosten als Zielfunktion | Bell (2006); Androutsopoulos & Zografos (2012) |
| Risikoparameter $\alpha=0{,}40$ (Bevölkerung), $\beta=0{,}40$ (Unfälle) | Cuneo et al. (2018) |
| Naturschutzgebiete als dritter Risikofaktor $\gamma$ | List & Mirchandani (1988), als methodische Weiterentwicklung gefordert |
| Gefahrgutklassen-Multiplikator $\mu_k$ | Bula et al. (2016) |
| Vorab-Screening / Graphreduktion vor Solver-Einsatz | List & Mirchandani (1988); Toth & Vigo (2002) |
| Rückfahrt risikofrei (Risiko als Funktion der Ladungsmenge) | List & Mirchandani (1988); Bula et al. (2016) |
| VRP-Struktur mit mehreren Kunden pro Tour | zweite ScienceDirect-Quelle (bikriterielles Routing & Scheduling); Bula et al. (2016) |
| Notwendigkeit gemeinsamer Routing/Scheduling-Optimierung | Bell (2006); zweite ScienceDirect-Quelle |
| Robuster, deterministischer Score statt Wahrscheinlichkeitsrechnung bei Low-Probability-High-Consequence-Ereignissen | Bell (2006) |
| Risikostreuung durch heterogene Flotte / Multi-Trip | Bell (2006), Mixed-Route-Konzept |
| Wechsel auf Metaheuristik bei großen Instanzen | Androutsopoulos & Zografos (2012); Bula et al. (2016) |
| Distanzmatrix-Ansatz als Standardmethode | Toth & Vigo (2002) — als ergänzende Referenz empfohlen, da die genannten Hazmat-Paper den Ansatz nur indirekt stützen |
